# 03 — Box Library & Reconstruction Validation

Build composable ZX boxes from algorithm sub-components,
compose them, and validate that compositions match monolithic circuits.

In [ ]:
import numpy as np
import pyzx as zx
from qiskit import QuantumCircuit

from zx_motifs.pipeline.composer import (
    compose_sequential,
    make_box_from_circuit,
    simplify_box,
)
from zx_motifs.algorithms.registry import make_grover, make_qft

## Grover Decomposition

Decompose Grover's algorithm into composable boxes:
**superposition → oracle → diffusion**

In [ ]:
n = 3

# Build sub-component boxes
def make_superposition_box(n_qubits):
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    return make_box_from_circuit(f"superposition_{n_qubits}q", qc, "superposition")


def make_grover_oracle_box(n_qubits, marked=0):
    qc = QuantumCircuit(n_qubits)
    binary = format(marked, f"0{n_qubits}b")
    for i, bit in enumerate(binary):
        if bit == "0":
            qc.x(i)
    if n_qubits >= 3:
        qc.h(n_qubits - 1)
        qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
        qc.h(n_qubits - 1)
    else:
        qc.cz(0, 1)
    for i, bit in enumerate(binary):
        if bit == "0":
            qc.x(i)
    return make_box_from_circuit(f"grover_oracle_{n_qubits}q", qc, "oracle")


def make_diffusion_box(n_qubits):
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    qc.x(range(n_qubits))
    qc.h(n_qubits - 1)
    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    qc.h(n_qubits - 1)
    qc.x(range(n_qubits))
    qc.h(range(n_qubits))
    return make_box_from_circuit(f"diffusion_{n_qubits}q", qc, "diffusion")


sup_box = make_superposition_box(n)
oracle_box = make_grover_oracle_box(n, marked=0)
diffusion_box = make_diffusion_box(n)

print(f"Superposition: {sup_box.graph.num_vertices()}V, {sup_box.graph.num_edges()}E")
print(f"Oracle: {oracle_box.graph.num_vertices()}V, {oracle_box.graph.num_edges()}E")
print(f"Diffusion: {diffusion_box.graph.num_vertices()}V, {diffusion_box.graph.num_edges()}E")

In [ ]:
# Compose: superposition -> oracle -> diffusion
grover_composed = compose_sequential(sup_box, oracle_box)
grover_composed = compose_sequential(grover_composed, diffusion_box)

# Build monolithic Grover for comparison
grover_mono = make_grover(n_qubits=n, marked_state=0, n_iterations=1)
mono_box = make_box_from_circuit("grover_monolithic", grover_mono, "grover")

# Validate semantic equivalence
match = zx.compare_tensors(grover_composed.graph, mono_box.graph)
print(f"Grover reconstruction validated: {match}")

In [ ]:
# Simplification comparison
composed_simplified = simplify_box(grover_composed, level="interior_clifford")

print(f"Composed (raw):        V={grover_composed.graph.num_vertices()} "
      f"E={grover_composed.graph.num_edges()}")
print(f"Composed (simplified): V={composed_simplified.graph.num_vertices()} "
      f"E={composed_simplified.graph.num_edges()}")
print(f"Monolithic (raw):      V={mono_box.graph.num_vertices()} "
      f"E={mono_box.graph.num_edges()}")

## QFT Decomposition

The QFT decomposes naturally into butterfly stages.

In [ ]:
def make_qft_stage_box(n_qubits, target_qubit):
    """One stage of QFT: H on target, then controlled rotations."""
    qc = QuantumCircuit(n_qubits)
    qc.h(target_qubit)
    for j in range(target_qubit + 1, n_qubits):
        angle = np.pi / (2 ** (j - target_qubit))
        qc.cp(angle, j, target_qubit)
    return make_box_from_circuit(
        f"qft_stage_q{target_qubit}_of_{n_qubits}", qc, "qft_butterfly"
    )


n_qft = 4
stages = [make_qft_stage_box(n_qft, i) for i in range(n_qft)]

# Compose QFT from stages
qft_composed = stages[0]
for stage in stages[1:]:
    qft_composed = compose_sequential(qft_composed, stage)

# Validate against monolithic QFT
qft_mono_box = make_box_from_circuit("qft_monolithic", make_qft(n_qft), "qft")
qft_valid = zx.compare_tensors(qft_composed.graph, qft_mono_box.graph)
print(f"QFT reconstruction validated: {qft_valid}")